# Beta estimators: FantasyGo selection model

Estimates how likely FantasyGo managers are to pick each player, per position and gameweek, for the opponent
side of the MVT (`mvt.ipynb`). For player $j$ of a position in round $t$:

$$\alpha_{j,t} = \exp(\beta_0 + x_{j,t}^T\beta), \qquad c_t \sim \text{DirichletMultinomial}(n_t, \alpha_t)$$

* $c_{j,t}$: the number of FantasyGo entries that picked player $j$ in round $t$ (`fg_manager_picks.csv`),
  $n_t = \sum_j c_{j,t}$
* $x_{j,t}$: the player's standardized features, known before the deadline
* $\beta_0$: sets how concentrated the picks are; $\alpha_{j,t}/\sum_k\alpha_{k,t}$ is the expected pick share

One Dirichlet-multinomial per round (the rounds are separate choices), $\beta \sim N(0, 1)$, fitted with NUTS. As in
`fit_position_pipeline`: rank and de-duplicate the candidate features, fit, keep the features whose 94% HDI
excludes 0, refit.

One fit per gameweek in `FIT_ROUNDS` (6-38), on the rounds before it only, so nothing is fitted on the
gameweek it predicts. `league_fit_for_round` returns the latest fit at or before a gameweek.

What changed from the earlier version (kept in `beta estimators.backup.ipynb`):

* target: FantasyGo pick counts instead of FPL ownership (`selected_by_percent` normalized), which was both the
  wrong population and a near-copy of a feature (correlation 0.98)
* likelihood: one Dirichlet-multinomial per round instead of one Dirichlet over all rounds' rows (whose
  "probabilities" summed to the number of rounds, not 1)
* features: `log1p` of ownership and transfers (they reached 7-8 standard deviations and overflowed `exp`),
  last round's FantasyGo share added, `opp_elo_h`/`opp_elo_a` (copies of `elo_h`/`elo_a`) and `round` removed,
  rows filtered before scaling
* an intercept $\beta_0$ (the total concentration is not fixed by the features)
* PCA dropped: the components mixed in the target and are not used by the model
* candidates are only the players known at the deadline, also for fitting (no rows selected on this round's picks),
  and `fg_share_prev` comes from the previous round's picks (it was the previous *row's* share, an older round's
  for 935 rows after a blank gameweek)
* a power $\kappa$ sharpens the shares for the team generator (see below)

In [1]:
import warnings
from pathlib import Path

import arviz as az
import joblib
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
from scipy.stats import spearmanr


warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("../..")
ALPHA_DRAWS = Path("league_alpha_draws.pkl")  # alpha draws per gameweek for the team generator
N_DRAWS = 500  # posterior draws kept per fit
FITS = Path("league_models.pkl")  # cached fits: delete to refit

FIT_ROUNDS = list(range(6, 39))  # one fit per gameweek, on the rounds before it
FIRST_ROUND = 4  # rounds 1-3 have little history for the rolling features
POSITIONS = ["GK", "DEF", "MID", "FWD"]
SEED = 42

## Data

One row per player with a fixture and round: candidates are players with minutes in the last 5 rounds or picked
by a FantasyGo manager last round, both known at the deadline. Fitting uses the same candidates: adding the players
picked this round would select rows on the outcome (a player with no recent minutes is only in the data when someone
picked him). Picks of other players, and of players without a fixture that round (no features), are left out.

`fg_share_prev` is the player's share of his position's FantasyGo picks in the previous round, from the picks
themselves: a player without a row that round (no fixture) still has his picks counted, and a player with no picks
then has 0, not the share of an earlier round.

In [2]:
data = pd.read_csv(
    ROOT / "inputs/rolled_player_data_23-24_all_players_xp_fixed.csv", low_memory=False
)
fg_picks = pd.read_csv(ROOT / "optimizer/data/fg_manager_picks.csv")

counts = fg_picks.groupby(["gameweek", "player_id"]).size().rename("fg_picks")
data = data.merge(counts, left_on=["round", "fpl_id"], right_index=True, how="left")
data["fg_picks"] = data["fg_picks"].fillna(0)
data["fg_share"] = data["fg_picks"] / data.groupby(["round", "position"])[
    "fg_picks"
].transform("sum")

# Features known before the deadline
prev_counts = fg_picks.groupby(["gameweek", "position", "player_id"]).size()
prev_share = (
    (prev_counts / prev_counts.groupby(level=["gameweek", "position"]).transform("sum"))
    .rename("fg_share_prev")
    .reset_index()
    .assign(round=lambda d: d["gameweek"] + 1)
)
data = data.merge(
    prev_share[["round", "player_id", "fg_share_prev"]],
    left_on=["round", "fpl_id"],
    right_on=["round", "player_id"],
    how="left",
).drop(columns="player_id")
data["fg_share_prev"] = data["fg_share_prev"].fillna(0)
data["log_fg_share_prev"] = np.log(data["fg_share_prev"] + 1e-4)
data["log_selected"] = np.log1p(data["selected_by_percent"])
data["log_transfers_in"] = np.log1p(data["transfers_in"])
data["log_transfers_out"] = np.log1p(data["transfers_out"])
data["elo_diff"] = data["elo"] - data["opp_elo"]
data["difficulty"] = np.where(
    data["was_home"], data["team_h_difficulty"], data["team_a_difficulty"]
)

data = data[
    (data["round"] >= FIRST_ROUND)
    & ((data["minutes_rolling_5"] > 0) | (data["fg_share_prev"] > 0))
].reset_index(drop=True)
print(data.groupby("position").agg(rows=("fpl_id", "size"), picks=("fg_picks", "sum")))

          rows    picks
position               
DEF       4656  30211.0
FWD       1722  21299.0
GK         919   9083.0
MID       6164  41090.0


/tmp/ipykernel_894292/3891749272.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["fg_share"] = data["fg_picks"] / data.groupby(["round", "position"])[
/tmp/ipykernel_894292/3891749272.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["log_fg_share_prev"] = np.log(data["fg_share_prev"] + 1e-4)
/tmp/ipykernel_894292/3891749272.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once

### Candidate features

In [3]:
COMMON = [
    "value",
    "log_selected",
    "log_transfers_in",
    "log_transfers_out",
    "log_fg_share_prev",
    "win_prob",
    "elo_diff",
    "difficulty",
    "was_home",
    "xP_rolling_3",
    "total_points_rolling_3",
    "points_5_ewm",
    "minutes_rolling_3",
    "starts_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "creativity_rolling_3",
    "threat_rolling_3",
]
ATTACK = [
    "expected_goals_rolling_3",
    "expected_assists_rolling_3",
    "expected_goal_involvements_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
]
DEFENCE = ["expected_goals_conceded_rolling_3", "clean_sheets_rolling_3"]
CANDIDATES = {
    "GK": COMMON + DEFENCE + ["saves_rolling_3"],
    "DEF": COMMON + ATTACK + DEFENCE,
    "MID": COMMON + ATTACK,
    "FWD": COMMON + ATTACK,
}

## Model

`select_features_by_correlation`, `_hdi_overlap` and `prune_by_hdi` are the earlier helpers, unchanged.
`fit_dirmult` is the Dirichlet-multinomial: with $A_t = \sum_j \alpha_{j,t}$, its log-likelihood (up to a constant) is
$\sum_t [\ln\Gamma(A_t) - \ln\Gamma(n_t + A_t)] + \sum_{j,t} [\ln\Gamma(c_{j,t} + \alpha_{j,t}) - \ln\Gamma(\alpha_{j,t})]$,
computed for all rounds at once.

In [4]:
def select_features_by_correlation(
    df,
    candidate_feats,
    target_col="select_prob",
    redundancy_threshold=0.8,
    min_corr=0.0,
    verbose=True,
):
    """
    Automatic feature selection in two steps:
      1. Rank candidate_feats by |correlation| with target_col, descending
         -- strongest predictors of xP first.
      2. Walk down the ranked list, greedily keeping a feature only if its
         |correlation| with every already-selected feature is below
         redundancy_threshold. A feature that's individually predictive of
         xP but redundant with something already kept (i.e. it's not
         adding new information) gets dropped.

    This is a filter-based analogue of stepwise selection: it favors
    features that are both informative about the target and
    non-redundant with each other, without needing to actually fit the
    (expensive) Dirichlet model repeatedly.

    Parameters
    ----------
    df                    : DataFrame containing candidate_feats + target_col
    candidate_feats       : list of column names to consider
    target_col            : column to rank relevance against (e.g. 'xP')
    redundancy_threshold  : drop a feature if |corr| with an already-kept
                             feature exceeds this (0-1). Lower = stricter
                             (keeps fewer, more independent features).
    min_corr              : also require |corr| with target_col to exceed
                             this floor to be considered at all (0-1).
                             Default 0.0 = no relevance floor, matching a
                             pure rank-then-deduplicate selection.

    Returns
    -------
    X_feats : list of selected feature names, in original candidate order
    """

    df_valid = df[candidate_feats + [target_col]].dropna()
    if len(df_valid) < len(df):
        dropped = len(df) - len(df_valid)
        if verbose:
            print(f"  dropped {dropped} rows with NaNs before computing correlations")

    target_corr = df_valid[candidate_feats].corrwith(df_valid[target_col]).abs()

    ranked = target_corr[~target_corr.index.duplicated(keep="first")].sort_values(
        ascending=False
    )  # target_corr.sort_values(ascending=False)

    feat_corr = df_valid[candidate_feats].corr().abs()

    selected = []
    # print(ranked.index)
    for feat in ranked.index:
        if pd.isna(ranked[feat]):
            if verbose:
                print(
                    f"  skipping {feat!r} -- undefined correlation with {target_col} "
                    f"(likely constant column)"
                )
            continue
        if ranked[feat] < min_corr:
            if verbose:
                print(
                    f"  skipping {feat!r} -- |corr| with {target_col} "
                    f"({ranked[feat]:.3f}) below min_corr={min_corr}"
                )
            continue
        if not selected:
            selected.append(feat)
            continue
        max_corr_with_selected = feat_corr.loc[feat, selected].max()
        if max_corr_with_selected < redundancy_threshold:
            selected.append(feat)
        elif verbose:
            worst = feat_corr.loc[feat, selected].idxmax()
            print(
                f"  dropping {feat!r} (|corr| with {target_col}={target_corr[feat]:.3f}) "
                f"-- |corr|={max_corr_with_selected:.3f} with already-selected {worst!r}"
            )

    # restore original candidate order for readability/downstream indexing
    X_feats = [f for f in candidate_feats if f in selected]

    if verbose:
        print(
            f"\nSelected {len(X_feats)} / {len(candidate_feats)} features "
            f"(ranked by |corr| with {target_col!r}, redundancy_threshold={redundancy_threshold}, "
            f"min_corr={min_corr}):"
        )
        print(X_feats)

    return X_feats


def _hdi_overlap(lo1, hi1, lo2, hi2):
    """
    Fraction of the *narrower* interval's width that's covered by the
    overlap with the other interval. 1.0 = one interval fully contains
    (or exactly equals) the other; 0.0 = no overlap at all.
    """
    overlap = max(0.0, min(hi1, hi2) - max(lo1, lo2))
    narrower_width = min(hi1 - lo1, hi2 - lo2)
    if narrower_width <= 0:
        return 0.0
    return overlap / narrower_width


def prune_by_hdi(summary, feats, overlap_threshold=0.8, verbose=True):
    """
    Keep only features whose 94% HDI excludes zero, AND deduplicate
    features whose HDIs are near-identical to one another (a sign the
    two are capturing essentially the same effect -- often leftover
    collinearity that survived the raw-feature correlation check,
    since it can also arise from *combinations* of correlated features
    rather than a single pairwise correlation).

    Greedy strategy: process candidates in order of decreasing |mean|
    (strongest effect first). A candidate is dropped if its HDI overlaps
    an already-kept feature's HDI by more than `overlap_threshold`
    (as a fraction of the narrower interval's width) -- i.e. the weaker
    of the two near-duplicate effects is the one that goes.

    Returns (kept_feats, kept_indices).
    """
    mask = (summary["hdi_3%"] > 0) | (summary["hdi_97%"] < 0)
    sig = summary[mask].copy()
    labels = sig.index.tolist()
    indices = [int(lbl.split("[")[1].split("]")[0]) for lbl in labels]

    if len(sig) == 0:
        return [], []

    # process strongest effect first, so a redundant *weaker* duplicate
    # is the one dropped rather than an arbitrary member of the pair
    order = np.argsort(-sig["mean"].abs().to_numpy())

    kept_local = []
    for i in order:
        lo_i, hi_i = sig["hdi_3%"].iloc[i], sig["hdi_97%"].iloc[i]
        redundant_with = None
        for k in kept_local:
            lo_k, hi_k = sig["hdi_3%"].iloc[k], sig["hdi_97%"].iloc[k]
            if _hdi_overlap(lo_i, hi_i, lo_k, hi_k) > overlap_threshold:
                redundant_with = k
                break
        if redundant_with is None:
            kept_local.append(i)
        elif verbose:
            print(
                f"  dropping {feats[indices[i]]!r} (label {labels[i]}) -- "
                f"HDI near-duplicate of {feats[indices[redundant_with]]!r} "
                f"(label {labels[redundant_with]})"
            )

    kept_local = sorted(kept_local)  # restore original feature order
    kept_indices = [indices[i] for i in kept_local]
    kept_feats = [feats[i] for i in kept_indices]
    return kept_feats, kept_indices


def fit_dirmult(
    X, counts, round_idx, sigma=1.0, draws=1000, tune=1000, chains=4, random_seed=SEED
):
    """NUTS fit of counts_t ~ DirichletMultinomial(n_t, exp(b0 + X beta)), one per round."""
    n_rounds = round_idx.max() + 1
    n_t = np.bincount(round_idx, weights=counts, minlength=n_rounds)
    with pm.Model():
        intercept = pm.Normal("intercept", 0.0, 2.0)
        beta = pm.Normal("beta", 0.0, sigma, shape=X.shape[1])
        alpha = pt.exp(pt.clip(intercept + pt.dot(X, beta), -20, 20))
        A = pt.inc_subtensor(pt.zeros(n_rounds)[round_idx], alpha)
        pm.Potential(
            "dirmult",
            pt.sum(pt.gammaln(A) - pt.gammaln(n_t + A))
            + pt.sum(pt.gammaln(counts + alpha) - pt.gammaln(alpha)),
        )
        trace = pm.sample(
            draws,
            tune=tune,
            chains=chains,
            cores=chains,
            target_accept=0.9,
            random_seed=random_seed,
            progressbar=False,
        )
    summary = az.summary(trace, var_names=["beta"])
    n_div = int(trace.sample_stats["diverging"].sum())
    if n_div or summary["r_hat"].max() > 1.01:
        print(
            f"  WARNING: {n_div} divergences, max r_hat {summary['r_hat'].max():.3f} -- provisional fit"
        )
    return trace, summary


def standardize(df, feats):
    X = df[feats].to_numpy(dtype=float)
    mean, std = X.mean(axis=0), X.std(axis=0)
    std[std == 0] = 1.0
    return (X - mean) / std, mean, std


def fit_position_pipeline(
    pos, fit_round, redundancy_threshold=0.7, min_corr=0.05, verbose=True
):
    """Select features, fit, prune by HDI and refit, on the position's rounds before fit_round."""
    train = data[(data["position"] == pos) & (data["round"] < fit_round)]
    if verbose:
        print(
            f"\n{'=' * 60}\n{pos}, rounds {train['round'].min()}-{train['round'].max()}, "
            f"{len(train)} rows\n{'=' * 60}"
        )
    feats = select_features_by_correlation(
        train,
        CANDIDATES[pos],
        target_col="fg_share",
        redundancy_threshold=redundancy_threshold,
        min_corr=min_corr,
        verbose=verbose,
    )
    counts = train["fg_picks"].to_numpy(dtype=float)
    round_idx = pd.factorize(train["round"])[0]

    X, _, _ = standardize(train, feats)
    trace, summary = fit_dirmult(X, counts, round_idx)
    feats_kept, _ = prune_by_hdi(summary, feats, verbose=verbose)
    if not feats_kept:
        print("WARNING: no features survived pruning -- keeping the full set")
        feats_kept = feats
    if verbose:
        print(f"Kept {len(feats_kept)} / {len(feats)}: {feats_kept}")

    X, mean, std = standardize(train, feats_kept)
    trace_final, summary_final = fit_dirmult(X, counts, round_idx)
    summary_final.index = feats_kept
    return {
        "feats": feats_kept,
        "X_mean": mean,
        "X_std": std,
        "summary_final": summary_final,
        "intercept": float(trace_final.posterior["intercept"].mean()),
        "beta_mean": trace_final.posterior["beta"].mean(dim=("chain", "draw")).values,
        # N_DRAWS evenly spaced posterior draws, for the team generator
        "intercept_draws": thin(trace_final.posterior["intercept"].values.reshape(-1)),
        "beta_draws": thin(trace_final.posterior["beta"].values.reshape(-1, len(feats_kept))),
    }


def thin(draws):
    """N_DRAWS evenly spaced rows of the stacked posterior draws."""
    return draws[np.linspace(0, len(draws) - 1, N_DRAWS).astype(int)]


def league_fit_for_round(fits, target_round):
    """The latest fit made at or before target_round."""
    available = [r for r in fits if r <= target_round]
    if not available:
        raise ValueError(f"No fit at or before round {target_round}")
    return fits[max(available)]


def alpha_draws(fit, X_new_raw):
    """alpha for new rows under each kept posterior draw: shape (N_DRAWS, rows)."""
    X = (X_new_raw - fit["X_mean"]) / fit["X_std"]
    eta = fit["intercept_draws"][:, None] + fit["beta_draws"] @ X.T
    return np.exp(np.clip(eta, -30, 30))


def alpha_league(fit, X_new_raw):
    """alpha for new rows, standardized with the fit's training mean and std."""
    X = (X_new_raw - fit["X_mean"]) / fit["X_std"]
    return np.exp(np.clip(fit["intercept"] + X @ fit["beta_mean"], -30, 30))

## One fit

In [5]:
example = fit_position_pipeline("MID", 20)
example["summary_final"][["mean", "sd", "hdi_3%", "hdi_97%", "ess_bulk", "r_hat"]]


MID, rounds 4-19, 2729 rows
  dropping 'value' (|corr| with fg_share=0.670) -- |corr|=0.777 with already-selected 'log_selected'
  dropping 'log_fg_share_prev' (|corr| with fg_share=0.574) -- |corr|=0.810 with already-selected 'log_selected'
  dropping 'expected_goal_involvements_rolling_3' (|corr| with fg_share=0.558) -- |corr|=0.771 with already-selected 'points_5_ewm'
  dropping 'xP_rolling_3' (|corr| with fg_share=0.538) -- |corr|=0.886 with already-selected 'points_5_ewm'
  dropping 'log_transfers_in' (|corr| with fg_share=0.535) -- |corr|=0.810 with already-selected 'points_5_ewm'
  dropping 'ict_index_rolling_3' (|corr| with fg_share=0.531) -- |corr|=0.864 with already-selected 'points_5_ewm'
  dropping 'threat_rolling_3' (|corr| with fg_share=0.522) -- |corr|=0.754 with already-selected 'points_5_ewm'
  dropping 'total_points_rolling_3' (|corr| with fg_share=0.520) -- |corr|=0.942 with already-selected 'points_5_ewm'
  dropping 'influence_rolling_3' (|corr| with fg_share=0.478

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [intercept, beta]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 14 seconds.


Initializing NUTS using jitter+adapt_diag...


Kept 6 / 7: ['log_selected', 'win_prob', 'difficulty', 'points_5_ewm', 'expected_goals_rolling_3', 'expected_assists_rolling_3']


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [intercept, beta]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 13 seconds.


,mean,sd,hdi_3%,hdi_97%,ess_bulk,r_hat
log_selected,0.696,0.030,0.639,0.750,2554.0,1.0
win_prob,0.229,0.037,0.163,0.303,3088.0,1.0
difficulty,-0.260,0.040,-0.335,-0.187,3388.0,1.0
points_5_ewm,0.611,0.033,0.549,0.671,2927.0,1.0
expected_goals_rolling_3,-0.039,0.020,-0.077,-0.003,2976.0,1.0
expected_assists_rolling_3,0.065,0.019,0.028,0.101,4067.0,1.0


## Every position and gameweek

Cached in `league_models.pkl` (delete it to refit).

In [6]:
if FITS.exists():
    league_models = joblib.load(FITS)
else:
    league_models = {
        pos: {r: fit_position_pipeline(pos, r, verbose=False) for r in FIT_ROUNDS}
        for pos in POSITIONS
    }
    joblib.dump(league_models, FITS)

pd.DataFrame(
    {
        (pos, r): {
            "features": len(f["feats"]),
            "max_r_hat": f["summary_final"]["r_hat"].max(),
            "min_ess": f["summary_final"]["ess_bulk"].min(),
        }
        for pos, fits in league_models.items()
        for r, f in fits.items()
    }
).T

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [intercept, beta]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 3 seconds.


Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


... (1,047 outputs left out)

Multiprocess sampling (4 chains in 4 jobs)


NUTS: [intercept, beta]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 8 seconds.


features  max_r_hat  min_ess
GK  6        3.0        1.0   2840.0
    7        4.0        1.0   3577.0
    8        3.0        1.0   3238.0
    9        3.0        1.0   3563.0
    10       5.0        1.0   3434.0
...          ...        ...      ...
FWD 34       5.0        1.0   3048.0
    35       5.0        1.0   3015.0
    36       6.0        1.0   2441.0
    37       6.0        1.0   2491.0
    38       6.0        1.0   2379.0

[132 rows x 3 columns]

In [7]:
# Coefficients of the last fit per position (standardized features)
pd.concat(
    {
        pos: fits[max(fits)]["summary_final"][["mean", "hdi_3%", "hdi_97%"]]
        for pos, fits in league_models.items()
    }
)

mean  hdi_3%  hdi_97%
GK  value                                 0.189   0.131    0.245
    log_transfers_in                      1.240   1.138    1.340
    win_prob                              0.203   0.126    0.289
    difficulty                           -0.294  -0.382   -0.202
DEF log_selected                          0.560   0.514    0.601
    log_fg_share_prev                     0.291   0.240    0.339
    win_prob                              0.421   0.368    0.472
    difficulty                           -0.183  -0.238   -0.124
    influence_rolling_3                   0.221   0.178    0.261
    xGChain_rolling_3                     0.085   0.050    0.117
    expected_goals_conceded_rolling_3     0.122   0.078    0.168
MID log_selected                          0.808   0.771    0.845
    win_prob                              0.265   0.215    0.317
    difficulty                           -0.109  -0.159   -0.059
    minutes_rolling_3                     0.486   0.431    0.543
    expected_goal_involvements_rolling_3  0.122   0.094    0.152
FWD log_selected                          0.442   0.359    0.516
    log_fg_share_prev                     0.386   0.304    0.470
    win_prob                              0.236   0.165    0.308
    difficulty                           -0.183  -0.256   -0.108
    points_5_ewm                          0.330   0.269    0.383
    creativity_rolling_3                  0.082   0.036    0.131

## Out-of-sample check

For every gameweek from the first fit round, the expected pick share $\alpha_j/\sum_k\alpha_k$ from that gameweek's
fit, against the actual FantasyGo share. Compared with three baselines: every candidate equally likely, FPL ownership
(`selected_by_percent`) and last round's FantasyGo share.

* `log_loss`: mean negative log-probability of each actual pick (lower is better)
* `spearman`: rank correlation of the predicted and actual shares
* `top10`: how many of the 10 most-picked players are in the predicted top 10

In [8]:
def scores(pred_share, g):
    p = np.clip(pred_share, 1e-6, None)
    p = p / p.sum()
    top = set(g.nlargest(10, "fg_picks").index)
    return {
        "log_loss": -(g["fg_picks"] * np.log(p)).sum() / g["fg_picks"].sum(),
        "spearman": spearmanr(p, g["fg_share"]).statistic,
        "top10": len(top & set(pd.Series(p, index=g.index).nlargest(10).index)),
    }


rows = []
for (pos, rnd), g in data[data["round"] >= min(FIT_ROUNDS)].groupby(
    ["position", "round"]
):
    fit = league_fit_for_round(league_models[pos], rnd)
    alpha = alpha_league(fit, g[fit["feats"]].to_numpy(dtype=float))
    for name, share in {
        "model": alpha / alpha.sum(),
        "uniform": np.ones(len(g)),
        "fpl_ownership": g["selected_by_percent"].to_numpy(dtype=float),
        "last_round_fg": g["fg_share_prev"].to_numpy(dtype=float),
    }.items():
        rows.append(
            {"position": pos, "round": rnd, "predictor": name, **scores(share, g)}
        )

check = pd.DataFrame(rows)
check.groupby(["position", "predictor"])[
    ["log_loss", "spearman", "top10"]
].mean().round(3).unstack("position")

/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input 

/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input 

/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input 

/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,


/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input 

/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input 

... (6 outputs left out)

/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input 

/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input 

/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,
/tmp/ipykernel_894292/3798005499.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(p, g["fg_share"]).statistic,


log_loss                      spearman                       \
position           DEF    FWD     GK    MID      DEF    FWD     GK    MID   
predictor                                                                   
fpl_ownership    3.963  2.920  3.073  3.628    0.597  0.696  0.552  0.661   
last_round_fg    5.429  4.059  5.173  4.799    0.532  0.641  0.416  0.660   
model            3.792  2.769  2.545  3.650    0.684  0.760  0.759  0.674   
uniform          4.919  3.912  3.268  5.204      NaN    NaN    NaN    NaN   

               top10                       
position         DEF    FWD     GK    MID  
predictor                                  
fpl_ownership  4.758  6.303  6.424  6.455  
last_round_fg  4.061  5.879  5.788  5.182  
model          5.970  7.394  8.030  6.515  
uniform        2.545  3.364  3.636  1.909

## Estimates for gameweeks 6-38

Each candidate's $\alpha$ and expected pick share (share of its position's picks) from that gameweek's fit, next to the
actual FantasyGo share. Written to `league_select_probs.csv` for the MVT.

In [9]:
estimates = []
for (pos, rnd), g in data[data["round"].isin(FIT_ROUNDS)].groupby(
    ["position", "round"]
):
    fit = league_fit_for_round(league_models[pos], rnd)
    alpha = alpha_league(fit, g[fit["feats"]].to_numpy(dtype=float))
    estimates.append(
        g[
            [
                "round",
                "position",
                "fpl_id",
                "name",
                "team",
                "value",
                "fg_picks",
                "fg_share",
            ]
        ].assign(alpha=alpha, pred_share=alpha / alpha.sum())
    )
estimates = pd.concat(estimates, ignore_index=True).sort_values(
    ["round", "position", "pred_share"], ascending=[True, True, False]
)
estimates.to_csv("league_select_probs.csv", index=False)
print(
    f"{len(estimates)} rows, gameweeks {estimates['round'].min()}-{estimates['round'].max()}"
)
estimates[estimates["round"] == 38].groupby("position").head(5)

13192 rows, gameweeks 6-38


,round,position,fpl_id,name,team,value,fg_picks,fg_share,alpha,pred_share
4541,38,DEF,616,Joško Gvardiol,Man City,52,133.0,0.159664,9.800311,0.128279
4423,38,DEF,20,William Saliba,Arsenal,59,45.0,0.054022,5.627704,0.073663
4515,38,DEF,511,Cristian Romero,Spurs,51,17.0,0.020408,4.703662,0.061568
4427,38,DEF,29,Benjamin White,Arsenal,61,98.0,0.117647,4.675479,0.061199
4513,38,DEF,506,Pedro Porro,Spurs,59,101.0,0.121248,4.384214,0.057386
6212,38,FWD,355,Erling Haaland,Man City,143,177.0,0.318345,9.704830,0.296432
6198,38,FWD,211,Nicolas Jackson,Chelsea,70,56.0,0.100719,4.504158,0.137579
6213,38,FWD,415,Alexander Isak,Newcastle,84,46.0,0.082734,3.627463,0.110800
6211,38,FWD,343,Julián Álvarez,Man City,64,4.0,0.007194,1.595838,0.048745
6187,38,FWD,60,Ollie Watkins,Aston Villa,89,34.0,0.061151,1.356195,0.041425


## Sharpening the pick shares

The fitted shares $\alpha_j/\sum_k\alpha_k$ are too flat: the simulated field spread its picks over about three times
as many players as the real one (the five most-picked players had 36% of the picks against 70%). A power $\kappa$
sharpens them, $\alpha_j^\kappa/\sum_k\alpha_k^\kappa$, fitted per position and gameweek by the log loss of the actual
picks in the rounds before it (from `FIT_ROUNDS[0]`, each with its own out-of-sample fit), so it is known at the
deadline. $\kappa = 1$ until a round is available. The team generator uses it.

In [10]:
from scipy.optimize import minimize_scalar


def pick_log_loss(power, rounds):
    """Mean negative log-probability of the actual picks under shares alpha^power / sum."""
    total, n = 0.0, 0.0
    for alpha, picks in rounds:
        log_p = power * np.log(alpha)
        log_p -= np.logaddexp.reduce(log_p)
        total -= (picks * log_p).sum()
        n += picks.sum()
    return total / n


by_round = {
    (pos, rnd): (g["alpha"].to_numpy(), g["fg_picks"].to_numpy())
    for (pos, rnd), g in estimates.groupby(["position", "round"])
}
ALPHA_POWER = {}
for pos in POSITIONS:
    for rnd in FIT_ROUNDS:
        earlier = [by_round[pos, r] for r in FIT_ROUNDS if r < rnd and (pos, r) in by_round]
        ALPHA_POWER[pos, rnd] = (
            minimize_scalar(pick_log_loss, bounds=(0.1, 10.0), args=(earlier,), method="bounded").x
            if earlier
            else 1.0
        )
pd.Series(ALPHA_POWER).unstack(0)[POSITIONS].round(2)


,GK,DEF,MID,FWD
6,1.00,1.00,1.00,1.00
7,0.90,0.98,1.10,0.96
8,0.86,1.09,1.17,1.02
9,0.93,0.92,1.19,0.97
10,1.02,0.93,1.12,0.96
11,1.05,0.96,1.12,0.99
12,1.00,0.98,1.08,0.97
13,1.03,0.99,1.07,0.98
14,1.08,1.00,1.08,0.98
15,1.10,1.01,1.08,0.96


## Alpha draws for the team generator

For every gameweek and position, the candidates known at the deadline (minutes in the last 5 rounds or picked
last round) and their $\alpha$ under each of the `N_DRAWS` posterior draws of that gameweek's fit. The team generator in
`mvt.ipynb` draws one of them per simulated opponent, as Algorithm 1 draws $\beta$, and sharpens it with `power` ($\kappa$).

In [11]:
draws = {}
for (rnd, pos), g in data[data["round"].isin(FIT_ROUNDS)].groupby(["round", "position"]):
    fit = league_fit_for_round(league_models[pos], rnd)
    draws.setdefault(rnd, {})[pos] = {
        "fpl_id": g["fpl_id"].to_numpy(),
        "alpha": alpha_draws(fit, g[fit["feats"]].to_numpy(dtype=float)).astype(np.float32),
        "power": ALPHA_POWER[pos, rnd],
    }
joblib.dump(draws, ALPHA_DRAWS)
print({pos: len(v["fpl_id"]) for pos, v in draws[38].items()}, "candidates in GW38;",
      f"{len(draws)} gameweeks written to {ALPHA_DRAWS}")

{'DEF': 147, 'FWD': 54, 'GK': 27, 'MID': 188} candidates in GW38; 33 gameweeks written to league_alpha_draws.pkl
